### 1. Imports and Configuration

In [6]:
import sys
import asyncio
import logging
from pathlib import Path

from IPython.display import display, Markdown

# -----------------------------------------------------------------------
# Path setup 
# -----------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from google.adk.agents                   import Agent
from google.adk.agents.sequential_agent  import SequentialAgent
from google.adk.agents.parallel_agent   import ParallelAgent
from google.adk.agents.loop_agent        import LoopAgent
from google.genai                        import types

from config import MODEL, APP_NAME, USER_ID, make_runner, strip_emojis
from tools  import (MORTGAGE_TOOLS, RISK_TOOLS, LOAN_TOOLS,
                     INVESTMENT_TOOLS, ALL_TOOLS)

logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger(__name__)

# The google-genai SDK warning that the response contains both text and function_call parts.
# If you access .text directly it only returns the text portion. 
# ADK doesn't use .text, it iterates all parts properly, so the function calls are handled correctly.
logging.getLogger("google.genai").setLevel(logging.ERROR)

### 2. Runner Helper

In [2]:
# -----------------------------------------------------------------------
# Helper to send a query, collect all events, and return the final
# response text. Separates thinking (part.thought=True) from content.
# Displays only the last final response to avoid duplicates when
# the model retries a failed tool call.
#
# ADK event model:
#   - Each LLM call, tool call, and tool response produces an Event.
#   - Events contain a Content object with a list of Part objects.
#   - Parts can be text, function_call, function_response, or thinking.
#   - Thinking parts have part.thought=True and carry the model's
#     internal reasoning (Gemma 4 <|channel>thought blocks).
#   - is_final_response() marks the event where the agent produces
#     its user-facing answer (as opposed to intermediate tool calls).
#
# Session and state:
#   - A session tracks one conversation. It holds events (history)
#     and state (key-value pairs shared across agents).
#   - initial_state pre-populates state keys that agents reference
#     via {key} in their instructions (needed for LoopAgent where
#     the first iteration reads keys not yet written).
#   - session_id can be passed to resume an existing session.
#   - user_id identifies the user within the session service.
# -----------------------------------------------------------------------

async def run_agent(agent, query, initial_state=None, session_id=None):
    """Send a single query to an agent and return the final response."""
    runner  = make_runner(agent)
    session = await runner.session_service.create_session(
        app_name = APP_NAME,
        user_id  = USER_ID,
        state    = initial_state or {},
    )
    sid = session_id or session.id

    # User message wrapped in ADK's Content/Part structure.
    # role="user" marks it as a user turn in the conversation.
    content = types.Content(
        role  = "user",
        parts = [types.Part(text=query)],
    )

    final_text    = None
    final_author  = None
    thinking_text = None

    # run_async yields events as the agent processes the query.
    # Multiple events are produced: one per LLM call, one per tool
    # call/response pair, and one or more final response events.
    async for event in runner.run_async(
        user_id     = USER_ID,
        session_id  = sid,
        new_message = content,
    ):
        # author identifies which agent produced this event
        # (relevant in multi-agent setups with sub-agents)
        author = getattr(event, "author", "unknown")

        # Log intermediate events: tool calls, tool responses, thinking
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    log.info("[%s] tool_call: %s(%s)",
                             author, part.function_call.name,
                             dict(part.function_call.args or {}))
                if hasattr(part, "function_response") and part.function_response:
                    log.info("[%s] tool_response: %s -> %s",
                             author, part.function_response.name,
                             str(part.function_response.response))
                if hasattr(part, "text") and part.text and getattr(part, "thought", False):
                    log.info("[%s] thinking: %s", author, part.text[:200])

        # Collect final response parts -- overwrite on each final event
        # so only the last complete response is kept (avoids duplicates
        # when the agent retries after a failed tool call)
        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if not hasattr(part, "text") or not part.text:
                    continue
                if getattr(part, "thought", False):
                    thinking_text = part.text
                else:
                    final_text   = strip_emojis(part.text)
                    final_author = author

    # Display once after all events are processed
    if thinking_text:
        display(Markdown(f"**Thinking ({final_author}):**\n\n{thinking_text}"))
    if final_text:
        display(Markdown(f"**{final_author}:**\n\n{final_text}"))

    return final_text

### 3. LlmAgent -- Single Agent with Tools

In [3]:
# -----------------------------------------------------------------------
# Basic LlmAgent backed by Gemma 4 E4B. Has access to all financial
# tools and decides which to call based on the user query.
# -----------------------------------------------------------------------

financial_advisor = Agent(
    model       = MODEL,
    name        = "financial_advisor",
    description = "General-purpose financial advisor with access to mortgage, loan, risk, and investment tools.",
    instruction = (
        "You act as a financial advisor. Always answer professionally."
        "Use the available tools to answer "
        "the user's question with concrete numbers."
        ""
    ),
    tools = ALL_TOOLS,
)

In [4]:
response = await run_agent(
    financial_advisor,
    "What would the monthly payment be on a 350k house with 70k down at 6.25% for 30 years?",
)
log.info("done")

03:21:53  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:21:55  INFO      Response received from the model.
03:21:55  WARNING   Warning: there are non-text parts in the response: ['function_call'], returning concatenated text result from text parts. Check the full candidates.content.parts accessor to get the full model response.
03:21:55  INFO      [financial_advisor] tool_call: calculate_monthly_payment({'principal': 350000, 'annual_rate': 6.25, 'down_payment': 70000, 'term_years': 30})
03:21:55  INFO      [financial_advisor] tool_response: calculate_monthly_payment -> {'principal': 350000, 'down_payment': 70000, 'loan_amount': 280000, 'annual_rate': 6.25, 'term_years': 30, 'monthly_payment': 1724.01, 'total_paid': 620642.94, 'total_interest': 340642.94, 'loan_to_value': 0.8}
03:21:55  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:21:57  INFO    

**financial_advisor:**

For a $350,000 home with a $70,000 down payment (20%), your loan amount would be **$280,000**. 

At an annual interest rate of **6.25%** over a **30-year** term, your estimated monthly principal and interest payment would be **$1,724.01**.

Over the full life of the loan, you would pay a total of approximately **$620,643**, which includes **$340,643** in total interest. Please keep in mind that this estimate does not include other monthly costs like property taxes, homeowners insurance, or private mortgage insurance (PMI), though with a 20% down payment, you would typically avoid PMI.

03:21:57  INFO      done


In [7]:
# -----------------------------------------------------------------------
# Multi-tool query -- the agent should chain affordability + eligibility.
# -----------------------------------------------------------------------

response = await run_agent(
    financial_advisor,
    "I earn 110k/year, have 400/month in debts, credit score 730. "
    "What is the max home I can afford at 6.5%? Am I eligible for a conventional loan "
    "if I want to buy a 450k house with 50k down?",
)
log.info("done")

03:24:12  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:24:14  INFO      Response received from the model.
03:24:14  INFO      [financial_advisor] tool_call: calculate_affordability({'annual_income': 110000, 'annual_rate': 6.5, 'monthly_debts': 400})
03:24:14  INFO      [financial_advisor] tool_call: check_loan_eligibility({'credit_score': 730, 'loan_amount': 400000, 'monthly_debts': 400, 'annual_income': 110000, 'property_value': 450000})
03:24:14  INFO      [financial_advisor] tool_response: calculate_affordability -> {'annual_income': 110000, 'monthly_debts': 400, 'max_monthly_mortgage_payment': 3541.67, 'max_loan_amount': 560329.99, 'max_home_price': 560329.99, 'down_payment': 0.0, 'annual_rate': 6.5, 'term_years': 30, 'resulting_dti': 43.0}
03:24:14  INFO      [financial_advisor] tool_response: check_loan_eligibility -> {'applicant_summary': {'credit_score': 730, 'estimated_dti': 31.94, 'ltv': 0.8889, 'is_first

**financial_advisor:**

Based on your annual income of $110,000 and current debts, here is a breakdown of your home-buying capacity and eligibility:

### **1. Maximum Affordability**
At a **6.5% interest rate** and a standard 30-year term, your maximum affordability is based on a standard debt-to-income (DTI) ratio of 43%:
*   **Maximum Monthly Mortgage Payment:** ~$3,542
*   **Maximum Loan Amount:** ~$560,330
*   **Maximum Home Price:** If you use your **$50,000 down payment**, you could technically afford a home priced up to approximately **$610,330**, assuming your taxes and insurance fit within that monthly budget.

### **2. Eligibility for a $450k House**
If you purchase a $450,000 home with a $50,000 down payment (loan amount of $400,000):
*   **Conventional Loan Eligibility:** **Yes, you are eligible.** Your credit score of 730 is well above the typical 620 minimum for conventional loans.
*   **Loan-to-Value (LTV):** Your LTV would be **88.9%**. Since this is higher than 80%, you will be required to pay Private Mortgage Insurance (PMI) until your equity reaches 20%.
*   **Debt-to-Income (DTI) Impact:** For this specific house, your projected DTI would be approximately **31.9%**, which is considered very healthy and comfortably below the 43% limit often used by lenders.

### **Summary of Options**
| Program | Eligible | Notes |
| :--- | :--- | :--- |
| **Conventional** | **YES** | Good option; requires PMI with your 11.1% down payment. |
| **FHA** | **YES** | Available with as little as 3.5% down, but involves permanent mortgage insurance. |
| **VA** | **NO** | Only available to veterans or active military members. |

**Recommendation:** Since you qualify for a conventional loan and have a solid credit score, this is likely your best path as it allows you to eventually cancel your mortgage insurance once you reach 20% equity, unlike FHA loans.

03:24:20  INFO      done


### 4. SequentialAgent -- Pipeline

In [8]:
# -----------------------------------------------------------------------
# Three-stage: mortgage calculation -> risk assessment -> summary.
# Each agent writes its result to session state via output_key,
# and the next agent reads it via {state_key} in its instruction.
# -----------------------------------------------------------------------

mortgage_agent = Agent(
    model       = MODEL,
    name        = "mortgage_calculator",
    description = "Calculates mortgage payments.",
    instruction = (
        "Calculate the monthly mortgage payment for the property described "
        "in the user query. Use the calculate_monthly_payment tool. "
        "Output only the raw tool result as text."
    ),
    tools      = MORTGAGE_TOOLS,
    output_key = "mortgage_result",
)

risk_agent = Agent(
    model       = MODEL,
    name        = "risk_assessor",
    description = "Evaluates loan risk.",
    instruction = (
        "Using the mortgage details in {mortgage_result}, assess the loan risk. "
        "Use the assess_loan_risk tool. Output only the raw tool result as text."
    ),
    tools      = RISK_TOOLS,
    output_key = "risk_result",
)

summary_agent = Agent(
    model       = MODEL,
    name        = "summariser",
    description = "Produces a concise summary.",
    instruction = (
        "Summarise the mortgage and risk results into a short recommendation. "
        "Mortgage: {mortgage_result}. Risk: {risk_result}. "
        "State the monthly payment, risk level, and whether to proceed."
    ),
)

sequential_pipeline = SequentialAgent(
    name        = "mortgage_pipeline",
    description = "Mortgage calculation to risk assessment to summary.",
    sub_agents  = [mortgage_agent, risk_agent, summary_agent],
)

response = await run_agent(
    sequential_pipeline,
    "500k house, 100k down, 6.0% rate, 30 year term. "
    "Buyer has 120k income, 600/mo debts, credit score 710.",
)
log.info("done")

03:25:42  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:25:43  INFO      Response received from the model.
03:25:43  INFO      [mortgage_calculator] tool_call: calculate_monthly_payment({'down_payment': 100000, 'term_years': 30, 'annual_rate': 6, 'principal': 500000})
03:25:43  INFO      [mortgage_calculator] tool_response: calculate_monthly_payment -> {'principal': 500000, 'down_payment': 100000, 'loan_amount': 400000, 'annual_rate': 6, 'term_years': 30, 'monthly_payment': 2398.2, 'total_paid': 863352.76, 'total_interest': 463352.76, 'loan_to_value': 0.8}
03:25:43  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:25:44  INFO      Response received from the model.
03:25:44  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:25:47  INFO      Response received from the model.
03:25:47  

**summariser:**

Based on the analysis, this loan is recommended for **approval** with a **LOW** risk level.

The monthly payment is **$2,398.20**. 

With a strong debt-to-income ratio of 30%, a solid 20% down payment (eliminating the need for PMI), and a healthy credit score, it is recommended to **proceed** with the mortgage.

03:25:52  INFO      done


### 5. ParallelAgent -- Concurrent Evaluation

In [9]:
# -----------------------------------------------------------------------
# Two independent analyses run concurrently: loan eligibility and
# investment growth projection. A sequential wrapper feeds both
# results into a final comparison agent.
# -----------------------------------------------------------------------

eligibility_agent = Agent(
    model       = MODEL,
    name        = "eligibility_checker",
    description = "Checks loan program eligibility.",
    instruction = (
        "Check loan eligibility for the applicant described in the user query. "
        "Use the check_loan_eligibility tool. Output the result."
    ),
    tools      = LOAN_TOOLS,
    output_key = "eligibility_result",
)

investment_agent = Agent(
    model       = MODEL,
    name        = "investment_projector",
    description = "Projects investment growth.",
    instruction = (
        "If the buyer invested the down payment amount mentioned in the user query "
        "instead of buying, project the growth over 10 years at 8% annual return "
        "using calculate_compound_interest. Output the result."
    ),
    tools      = INVESTMENT_TOOLS,
    output_key = "investment_result",
)

parallel_analysis = ParallelAgent(
    name        = "parallel_analysis",
    description = "Runs eligibility and investment projection concurrently.",
    sub_agents  = [eligibility_agent, investment_agent],
)

comparison_agent = Agent(
    model       = MODEL,
    name        = "buy_vs_invest",
    description = "Compares buying vs investing.",
    instruction = (
        "Compare the two options based on: "
        "Eligibility: {eligibility_result}. "
        "Investment alternative: {investment_result}. "
        "Give a short recommendation."
    ),
)

parallel_then_compare = SequentialAgent(
    name        = "parallel_then_compare",
    description = "Parallel analysis followed by comparison.",
    sub_agents  = [parallel_analysis, comparison_agent],
)

response = await run_agent(
    parallel_then_compare,
    "Applicant: 95k income, 700 credit score, 500/mo debts, "
    "wants a 380k house with 80k down. First-time buyer, not a veteran.",
)
log.info("done")

03:26:18  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:26:18  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:26:19  INFO      Response received from the model.
03:26:19  INFO      [investment_projector] tool_call: calculate_compound_interest({'annual_rate': 8, 'principal': 80000, 'years': 10})
03:26:19  INFO      [investment_projector] tool_response: calculate_compound_interest -> {'principal': 80000, 'annual_rate': 8, 'years': 10, 'monthly_contribution': 0.0, 'future_value': 177571.22, 'total_contributions': 80000.0, 'total_interest_earned': 97571.22, 'yearly_summary': [{'year': 1, 'balance': 86639.96, 'total_contributed': 80000.0, 'interest_earned': 6639.96}, {'year': 2, 'balance': 93831.03, 'total_contributed': 80000.0, 'interest_earned': 13831.03}, {'year': 3, 'balance': 101618.96, 'total_contributed': 80000.0, 'interest_earned': 21618.96}, {

**buy_vs_invest:**

Based on your profile, here is a comparison between purchasing the home and investing your capital:

### **1. Home Purchase: Eligibility & Financing**
*   **Best Path: Conventional Loan.** You are fully qualified.
*   **Equity Advantage:** With an **$80,000 down payment** (21% down), you reach a Loan-to-Value (LTV) of 78.95%. This allows you to **avoid Private Mortgage Insurance (PMI)**, significantly lowering your monthly obligation.
*   **Debt Health:** Your Debt-to-Income (DTI) ratio of ~30.3% is well below the typical 43-45% limit, placing you in a very strong borrowing position.
*   **Comparison:** While you qualify for FHA, the mandatory Mortgage Insurance Premium (MIP) makes it less cost-effective than a Conventional loan in your specific case.

### **2. Investment Alternative: Opportunity Cost**
*   **Capital Growth:** If you chose to invest the **$80,000** in a market account with an 8% annual return instead of buying, the funds would grow to approximately **$177,571.22** over 10 years.
*   **The Gain:** This represents a total interest gain of **$97,571.22**. 
*   **Trade-off:** By using the cash for a down payment, you are essentially "trading" that 8% market gain for the stability of fixed housing costs and the potential appreciation of the $380,000 property.

### **Recommendation**
**Proceed with the Home Purchase (Conventional Loan).** 
Because you can afford a 20%+ down payment, you are in the "sweet spot" of mortgage lending. Avoiding PMI and securing a primary residence provides immediate utility and forced savings through equity. However, keep in mind that your $80,000 is a high-performing asset; only buy if you plan to stay in the home long enough for property appreciation to compete with that ~8% market benchmark.

03:26:28  INFO      done


### 6. LoopAgent -- Iterative Refinement

In [10]:
# -----------------------------------------------------------------------
# Generator-critic loop: 
# One agent proposes a loan scenario, another critiques it.
# The generator reads {feedback} from state to adjust on each pass.
# -----------------------------------------------------------------------

loan_proposer = Agent(
    model       = MODEL,
    name        = "loan_proposer",
    description = "Proposes a loan structure.",
    instruction = (
        "Propose a loan structure (rate, term, down payment) for the buyer described "
        "in the user query. If there is previous feedback in {feedback}, adjust your "
        "proposal to address the concerns. Use compare_loans and calculate_monthly_payment "
        "to validate your proposal. Output the proposed structure and key numbers."
    ),
    tools      = MORTGAGE_TOOLS + LOAN_TOOLS,
    output_key = "proposal",
)

loan_critic = Agent(
    model       = MODEL,
    name        = "loan_critic",
    description = "Critiques a loan proposal.",
    instruction = (
        "Review the loan proposal in {proposal}. Use calculate_dti_ratio and "
        "assess_loan_risk to evaluate it. If the proposal is sound (DTI under 40% "
        "and risk level is LOW or MODERATE), output exactly 'APPROVED'. "
        "Otherwise output specific feedback on what to fix."
    ),
    tools      = RISK_TOOLS + LOAN_TOOLS,
    output_key = "feedback",
)

refinement_loop = LoopAgent(
    name           = "loan_refinement",
    description    = "Iterates proposal and critique until approved or max iterations.",
    sub_agents     = [loan_proposer, loan_critic],
    max_iterations = 3,
)

response = await run_agent(
    refinement_loop,
    "Buyer: 85k income, 750 credit, 1200/mo existing debts, "
    "looking at a 420k property. Wants the lowest possible monthly payment.",
    initial_state={"feedback": "", "proposal": ""},
)
log.info("done")

03:27:28  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:27:33  INFO      Response received from the model.
03:27:33  INFO      [loan_proposer] tool_call: calculate_affordability({'monthly_debts': 1200, 'term_years': 30, 'annual_income': 85000, 'annual_rate': 6.5})
03:27:33  INFO      [loan_proposer] tool_response: calculate_affordability -> {'annual_income': 85000, 'monthly_debts': 1200, 'max_monthly_mortgage_payment': 1845.83, 'max_loan_amount': 292030.8, 'max_home_price': 292030.8, 'down_payment': 0.0, 'annual_rate': 6.5, 'term_years': 30, 'resulting_dti': 43.0}
03:27:33  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:27:35  INFO      Response received from the model.
03:27:35  INFO      [loan_proposer] tool_call: compare_loans({'terms': [30], 'loan_amount': 336000, 'rates': [6, 6.5, 7]})
03:27:35  INFO      [loan_proposer] tool_response: compar

**loan_critic:**

APPROVED

03:28:16  INFO      done


### 7. Session State Inspection

In [15]:
# -----------------------------------------------------------------------
# Run the sequential pipeline again but inspect the session state
# after execution to see what each agent wrote via output_key.
# -----------------------------------------------------------------------

runner  = make_runner(sequential_pipeline)
session = await runner.session_service.create_session(
    app_name = APP_NAME,
    user_id  = USER_ID,
)

content = types.Content(
    role  = "user",
    parts = [types.Part(text=(
        "400k house, 80k down, 6.5% rate, 30yr. "
        "Buyer: 100k income, 500/mo debts, 720 credit."
    ))],
)

async for event in runner.run_async(
    user_id     = USER_ID,
    session_id  = session.id,
    new_message = content,
):
    pass

# Retrieve the session to inspect state written by each agent.
updated_session = await runner.session_service.get_session(
    app_name   = APP_NAME,
    user_id    = USER_ID,
    session_id = session.id,
)

log.info("session state keys: %s", list(updated_session.state.keys()))
for key in ["mortgage_result", "risk_result"]:
    value = updated_session.state.get(key, "<not set>")
    log.info("state[%s] = %s", key, str(value)[:300])

03:53:19  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:53:21  INFO      Response received from the model.
03:53:21  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:53:22  INFO      Response received from the model.
03:53:22  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:53:24  INFO      Response received from the model.
03:53:24  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:53:25  INFO      Response received from the model.
03:53:25  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
03:53:25  INFO      AFC is enabled with max remote calls: 10.
03:53:27  INFO      Response received from the model.
03:53:27  INFO      session state keys: ['mortga

In [16]:
# -----------------------------------------------------------------------
# Event log -- iterate over all events stored in the session
# to see the full trace of agent interactions.
# -----------------------------------------------------------------------

log.info("event count: %d", len(updated_session.events))
for i, ev in enumerate(updated_session.events):
    author = getattr(ev, "author", "?")
    parts_summary = []
    if ev.content and ev.content.parts:
        for p in ev.content.parts:
            if hasattr(p, "function_call") and p.function_call:
                parts_summary.append(f"call:{p.function_call.name}")
            elif hasattr(p, "function_response") and p.function_response:
                parts_summary.append(f"resp:{p.function_response.name}")
            elif hasattr(p, "text") and p.text:
                parts_summary.append(f"text:{p.text[:80]}")
    log.info("  event[%02d] author=%-25s parts=%s", i, author, parts_summary)

03:53:32  INFO      event count: 8
03:53:32  INFO        event[00] author=user                      parts=['text:400k house, 80k down, 6.5% rate, 30yr. Buyer: 100k income, 500/mo debts, 720 cre']
03:53:32  INFO        event[01] author=mortgage_calculator       parts=['call:calculate_monthly_payment']
03:53:32  INFO        event[02] author=mortgage_calculator       parts=['resp:calculate_monthly_payment']
03:53:32  INFO        event[03] author=mortgage_calculator       parts=["text:{'loan_amount': 320000.0, 'monthly_payment': 2022.62, 'total_paid': 728142.36, '"]
03:53:32  INFO        event[04] author=risk_assessor             parts=['call:assess_loan_risk']
03:53:32  INFO        event[05] author=risk_assessor             parts=['resp:assess_loan_risk']
03:53:32  INFO        event[06] author=risk_assessor             parts=["text:{'approval_recommendation': 'APPROVE', 'overall_risk_level': 'LOW', 'metrics': {"]
03:53:32  INFO        event[07] author=summariser                parts=['tex